# Démo : stratégies de découpage

**Contexte :** assistant du service client de Lagune Banque.

Un même texte est découpé selon quatre stratégies. Les couleurs distinguent les chunks et les zones de recouvrement ; elles montrent ce qui sera transmis au modèle.

**Sommaire**
- [Configuration](#configuration)
- [Textes de démonstration](#textes)
- [Visualisation des coupures](#visualisation)
- [Découpage à taille fixe](#taille-fixe)
- [Découpage récursif](#recursif)
- [Découpage par structure](#structure)
- [Découpage sémantique](#semantique)
- [Tableaux](#tableaux)
- [Synthèse](#synthese)

> **Source :** progression inspirée de « 5 Levels of Text Splitting » de Greg Kamradt. Outils de visualisation : <https://chunkviz.up.railway.app/> et <https://chunking.playground.weaviate.io/>.

## <a id="configuration"></a>Configuration

In [ ]:
import math
import re
import sys
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../starter/code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
from config import prepare

In [ ]:
prepare(embedding=True)

## <a id="textes"></a>Textes de démonstration

Deux extraits du corpus : l'article 4 des conditions générales (six clauses numérotées) et le tableau des frais de retard de la grille tarifaire. L'extraction d'un tableau de PDF donne une cellule par ligne.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
textes = {d["id"]: "\n".join(t for _, t in d["pages"]) for d in documents}

cg = textes["DOC-01"]
debut = cg.index("Article 4 –", cg.index("Article 4 –") + 10)
ARTICLE = cg[debut:cg.index("Article 5 –", debut)].strip()

grille = textes["DOC-06"]
debut = grille.index(" Produit\nFrais de retard")
fin = grille.index("Par échéance impayée", debut) + len("Par échéance impayée")
TABLEAU = grille[debut:fin].strip()

print(ARTICLE)
print("\n---\n")
print(TABLEAU)

## <a id="visualisation"></a>Visualisation des coupures

`show` retrouve la position de chaque chunk dans le texte et colore le texte chunk par chunk. Un petit numéro marque le début de chacun.

In [ ]:
COULEURS = ["#cfe8ff", "#d7f5d3", "#ffe7b8", "#f4d6ff"]


def show(texte, chunks, titre):
    zones, position = [], 0
    for p in chunks:
        position = texte.find(p, position)
        zones.append((position, position + len(p)))
    bornes = sorted({0, len(texte), *[x for zone in zones for x in zone]})
    debuts = {zone[0]: i for i, zone in enumerate(zones)}
    sortie = ""
    for a, b in zip(bornes, bornes[1:]):
        couvrant = [i for i, (d, f) in enumerate(zones) if d <= a < f]
        fond = "#ff9d9d" if len(couvrant) > 1 else COULEURS[couvrant[0] % 4] if couvrant else "#eeeeee"
        marque = f"<sup><b>{debuts[a]}</b></sup>" if a in debuts else ""
        sortie += f'{marque}<span style="background:{fond}">{escape(texte[a:b])}</span>'
    propres = sum(p.rstrip().endswith((".", "?", "!", ":", ";")) for p in chunks)
    display(HTML(f'<p><b>{titre}</b> : {len(chunks)} chunks, {propres} se terminent par une fin de phrase</p>'
                 f'<pre style="white-space:pre-wrap;line-height:1.5">{sortie}</pre>'))

## <a id="taille-fixe"></a>Découpage à taille fixe

`CharacterTextSplitter` de LangChain, avec `separator=""`, coupe sans regarder le texte : des fenêtres d'au plus 200 caractères, avec 30 caractères de recouvrement. Il compte des caractères, il ne lit pas le texte.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

fixe = CharacterTextSplitter(separator="", chunk_size=200, chunk_overlap=30)
show(ARTICLE, fixe.split_text(ARTICLE), "Taille fixe, 200 caractères")

Des chunks se coupent au milieu d'un mot (« initia ») ou d'un nombre (« 000 FCFA »), et la règle 4.2 (le plafond initial et sa progression) est coupée entre deux chunks. Une question sur le plafond ne retrouvera qu'une moitié de la règle.

## <a id="recursif"></a>Découpage récursif

Le splitter de LangChain essaie de couper aux séparateurs les plus forts d'abord : saut de paragraphe, saut de ligne, fin de phrase, espace. Même taille, même recouvrement.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursif = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""])
show(ARTICLE, recursif.split_text(ARTICLE), "Récursif, 200 caractères")

La clause 4.2 est ici entière, mais les clauses 4.5 et 4.6 sont coupées en deux chunks : la taille maximale prime sur le sens.

## <a id="structure"></a>Découpage par structure

Le texte porte déjà sa structure : chaque clause commence par un numéro (`4.1`, `4.2`). On coupe à chaque numéro, quelle que soit la taille. Le titre de l'article reste avec la première clause.

In [ ]:
debuts = [0] + [m.start() for m in re.finditer(r"(?m)^\d\.\d ", ARTICLE)][1:]
par_structure = [ARTICLE[a:b].strip() for a, b in zip(debuts, debuts[1:] + [len(ARTICLE)])]
show(ARTICLE, par_structure, "Par structure, une clause par chunk")

Chaque chunk est une règle complète : la clause 4.2 est entière. Les tailles varient (de 105 à 178 caractères). Cette méthode demande un texte qui a une structure, et un code adapté à chaque type de document : numéros d'articles, titres HTML, questions d'une FAQ.

## <a id="semantique"></a>Découpage sémantique

On calcule l'embedding de chaque phrase, puis on mesure la distance entre phrases voisines. Quand elle dépasse un seuil, le sujet change : on coupe. Le seuil est ici le 75e centile des distances observées.

In [ ]:
phrases = [m.span() for m in re.finditer(r"(?s)\S.*?[.?!](?=\s|$)", ARTICLE)]
vecteurs = rag.embeddings([ARTICLE[a:b] for a, b in phrases])
ecarts = [math.dist(v1, v2) ** 2 for v1, v2 in zip(vecteurs, vecteurs[1:])]
seuil = sorted(ecarts)[int(len(ecarts) * 0.75)]

pd.DataFrame({"phrase": [ARTICLE[a:b][:60].replace("\n", " ") for a, b in phrases[1:]],
              "écart avec la précédente": [round(e, 2) for e in ecarts],
              "coupure": ["✂" if e >= seuil else "" for e in ecarts]})

In [ ]:
coupures = [i for i, e in enumerate(ecarts, 1) if e >= seuil]
groupes = [(coupures[i - 1] if i else 0, coupures[i] if i < len(coupures) else len(phrases)) for i in range(len(coupures) + 1)]
semantique = [ARTICLE[phrases[a][0]:phrases[b - 1][1]] for a, b in groupes]
show(ARTICLE, semantique, "Sémantique, seuil au 75e centile")

Sur ce texte, le chunking sémantique est **moins bon** que le chunking par structure : il sépare le montant de la clause 4.1 de sa durée, parce que ces deux phrases parlent de choses différentes. Il convient mieux à des textes continus (comptes rendus, articles) sans numérotation. Il coûte aussi un appel d'embedding par phrase, et le seuil se règle à la main.

## <a id="tableaux"></a>Tableaux

L'extraction d'un tableau de PDF donne une cellule par ligne. Un chunking à taille fixe ne le sait pas.

In [ ]:
show(TABLEAU, CharacterTextSplitter(separator="", chunk_size=50, chunk_overlap=8).split_text(TABLEAU), "Taille fixe, 50 caractères")

Le prix « 2 000 FCFA » est coupé en deux : « 2 » reste avec « Prêt Express », « 000 FCFA » passe dans le chunk suivant, loin de son produit. Une question sur les frais de retard du Prêt Express ne trouve pas sa réponse entière dans un même chunk.

Découper par ligne du tableau (trois cellules par ligne) garde chaque produit avec son prix :

In [ ]:
cellules = TABLEAU.split("\n")
lignes = ["\n".join(cellules[i:i + 3]) for i in range(0, len(cellules), 3)]
show(TABLEAU, lignes, "Par ligne du tableau")

## <a id="synthese"></a>Synthèse

| Niveau | Coupe selon | Avantage | Limite |
|---|---|---|---|
| 1. Taille fixe | le nombre de caractères | trivial, rapide | coupe mots, phrases et règles |
| 2. Récursif | les séparateurs du texte, sous une taille maximale | bon point de départ | ignore le sens et les tableaux |
| 3. Structure | les titres, numéros, questions du document | chunks complets | un code par type de document |
| 4. Sémantique | les changements de sujet (embeddings) | utile sur du texte continu | un appel par phrase, seuil à régler |
| 5. Agentique | la décision d'un modèle de langage | le plus fin | lent et cher, non montré ici |

Le choix se mesure : le TP1 compare six configurations sur des questions dont la réponse doit tenir dans un seul chunk.